In [1]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import lightgbm as lgb


# ==============================================================================
# 0. CONFIGURATION
# ==============================================================================

SEED = 42
BATCH_SIZE = 64
EPOCHS = 150
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4

# --------------------------------------------------------------------------
# IMPORTANT COLUMN MAPPING
# --------------------------------------------------------------------------
# This code assumes the target array is ordered as:
#
#   targets[:, 0] -> Er_ohm-cm
#   targets[:, 1] -> UPV_m/s
#   targets[:, 2] -> Cs_Mpa
#   targets[:, 3] -> Ts_Mpa
#   targets[:, 4] -> Fs_Mpa
#
# If your NPZ target order is different, change ONLY these indices.
# --------------------------------------------------------------------------

NDT_TARGET_INDICES = [0, 1]
STRENGTH_TARGET_INDICES = [2, 3, 4]

NDT_TARGETS = ["Er_ohm-cm", "UPV_m/s"]
STRENGTH_TARGETS = ["Cs_Mpa", "Ts_Mpa", "Fs_Mpa"]

# Assumption: curing age is the LAST column in the input matrix.
# If not, change this to the correct zero-based column index.
AGE_INPUT_INDEX = -1

In [2]:
# ==============================================================================
# 1. DETERMINISTIC SEED & DEVICE SETUP
# ==============================================================================

def set_seed(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"[INFO] Device: {device}")

[INFO] Device: cuda


In [12]:
# ==============================================================================
# 2. LOAD PRE-SPLIT NPZ DATA
# ==============================================================================

def load_npz_data(
    train_path="ConcreteXAI/Model_Code/Data_train.npz",
    validation_path="ConcreteXAI/Model_Code/Data_validation.npz",
    test_path="ConcreteXAI/Model_Code/Data_test.npz"
):
    """
    Loads the already-created train, validation and test partitions.

    Each NPZ file must contain:
        inputs
        targets
    """

    train_npz = np.load(train_path)
    train_inputs = train_npz["inputs"].astype(np.float64)
    train_targets = train_npz["targets"].astype(np.float64)

    validation_npz = np.load(validation_path)
    valid_inputs = validation_npz["inputs"].astype(np.float64)
    valid_targets = validation_npz["targets"].astype(np.float64)

    test_npz = np.load(test_path)
    test_inputs = test_npz["inputs"].astype(np.float64)
    test_targets = test_npz["targets"].astype(np.float64)

    print("=" * 80)
    print("DATASET LOADED")
    print("=" * 80)

    print(f"Train inputs       : {train_inputs.shape}")
    print(f"Train targets      : {train_targets.shape}")
    print(f"Validation inputs  : {valid_inputs.shape}")
    print(f"Validation targets : {valid_targets.shape}")
    print(f"Test inputs        : {test_inputs.shape}")
    print(f"Test targets       : {test_targets.shape}")

    return (
        train_inputs,
        train_targets,
        valid_inputs,
        valid_targets,
        test_inputs,
        test_targets
    )

In [13]:
# ==============================================================================
# 3. DATA VALIDATION
# ==============================================================================

def validate_dataset_shapes(
    X_train,
    y_train,
    X_valid,
    y_valid,
    X_test,
    y_test
):
    """
    Basic consistency checks before training.
    """

    if X_train.ndim != 2 or X_valid.ndim != 2 or X_test.ndim != 2:
        raise ValueError("All input arrays must be 2-dimensional.")

    if y_train.ndim != 2 or y_valid.ndim != 2 or y_test.ndim != 2:
        raise ValueError("All target arrays must be 2-dimensional.")

    if X_train.shape[1] != X_valid.shape[1] or X_train.shape[1] != X_test.shape[1]:
        raise ValueError(
            "Train, validation and test input arrays must have the same number "
            "of features."
        )

    if y_train.shape[1] != y_valid.shape[1] or y_train.shape[1] != y_test.shape[1]:
        raise ValueError(
            "Train, validation and test target arrays must have the same "
            "number of target columns."
        )

    required_target_index = max(
        max(NDT_TARGET_INDICES),
        max(STRENGTH_TARGET_INDICES)
    )

    if y_train.shape[1] <= required_target_index:
        raise ValueError(
            f"The current configuration requires at least "
            f"{required_target_index + 1} target columns, but the dataset has "
            f"{y_train.shape[1]}."
        )

In [14]:
# ==============================================================================
# 4. PYTORCH MULTI-TASK DATASET
# ==============================================================================

class ConcreteMTLDataset(Dataset):
    """
    Dataset for multi-task regression.

    Missing target values are handled using a binary observation mask:
        mask = 1 -> target is available
        mask = 0 -> target is missing
    """

    def __init__(self, X: np.ndarray, y: np.ndarray):
        self.X = torch.tensor(X, dtype=torch.float32)

        masks = (~np.isnan(y)).astype(np.float32)
        cleaned_y = np.nan_to_num(y, nan=0.0)

        self.y = torch.tensor(cleaned_y, dtype=torch.float32)
        self.masks = torch.tensor(masks, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx], self.masks[idx]

In [15]:
# ==============================================================================
# 5. PHYSICS-INFORMED MULTI-TASK NETWORK
# ==============================================================================

class Mish(nn.Module):
    def forward(self, x):
        return x * torch.tanh(nn.functional.softplus(x))


class PhysicsInformedMTNN(nn.Module):
    """
    Shared representation network with independent regression heads for:
        1. Compressive strength
        2. Tensile strength
        3. Flexural strength
    """

    def __init__(
        self,
        input_dim: int,
        hidden_dim: int = 128,
        dropout_rate: float = 0.2
    ):
        super().__init__()

        self.shared_trunk = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            Mish(),
            nn.Dropout(dropout_rate),

            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.BatchNorm1d(hidden_dim // 2),
            Mish(),
            nn.Dropout(dropout_rate),

            nn.Linear(hidden_dim // 2, hidden_dim // 4),
            nn.BatchNorm1d(hidden_dim // 4),
            Mish()
        )

        trunk_out = hidden_dim // 4

        self.head_cs = nn.Linear(trunk_out, 1)
        self.head_ts = nn.Linear(trunk_out, 1)
        self.head_fs = nn.Linear(trunk_out, 1)

    def forward(self, x):
        features = self.shared_trunk(x)

        cs = self.head_cs(features)
        ts = self.head_ts(features)
        fs = self.head_fs(features)

        return torch.cat([cs, ts, fs], dim=1)

In [16]:
# ==============================================================================
# 6. PHYSICS-INFORMED LOSS
# ==============================================================================

def compute_physics_loss(
    preds: torch.Tensor,
    targets: torch.Tensor,
    masks: torch.Tensor,
    inputs: torch.Tensor,
    age_col_idx: int,
    lambda_ratio: float = 0.5,
    lambda_mono: float = 0.1
):
    """
    Composite objective:

    1. Masked empirical MSE
    2. Flexural strength <= 35% of compressive strength
    3. Tensile strength <= flexural strength
    4. Compressive strength should not decrease with curing age
    """

    # ----------------------------------------------------------------------
    # 1. Masked empirical loss
    # ----------------------------------------------------------------------

    squared_error = (preds - targets) ** 2
    masked_error = squared_error * masks

    empirical_loss = masked_error.sum() / (masks.sum() + 1e-8)

    cs = preds[:, 0]
    ts = preds[:, 1]
    fs = preds[:, 2]

    # ----------------------------------------------------------------------
    # 2. Mechanics-based ratio penalties
    # ----------------------------------------------------------------------

    ratio_penalty_1 = torch.relu(
        fs - 0.35 * cs
    ).pow(2).mean()

    ratio_penalty_2 = torch.relu(
        ts - fs
    ).pow(2).mean()

    ratio_loss = ratio_penalty_1 + ratio_penalty_2

    # ----------------------------------------------------------------------
    # 3. Monotonic curing-age penalty
    # ----------------------------------------------------------------------

    if inputs.requires_grad:

        grad_cs = torch.autograd.grad(
            outputs=cs.sum(),
            inputs=inputs,
            create_graph=True,
            retain_graph=True
        )[0]

        d_cs_d_age = grad_cs[:, age_col_idx]

        monotonicity_loss = torch.relu(
            -d_cs_d_age
        ).pow(2).mean()

    else:
        monotonicity_loss = torch.tensor(
            0.0,
            device=preds.device
        )

    total_loss = (
        empirical_loss
        + lambda_ratio * ratio_loss
        + lambda_mono * monotonicity_loss
    )

    return total_loss

In [17]:
# ==============================================================================
# 7. CONFORMAL PREDICTION CALIBRATOR
# ==============================================================================

class MultiTaskConformalCalibrator:

    def __init__(self, alpha: float = 0.05):
        self.alpha = alpha
        self.q_hat = {}

    def calibrate(self, y_true: np.ndarray, y_pred: np.ndarray):

        for i, target_name in enumerate(STRENGTH_TARGETS):

            valid_mask = ~np.isnan(y_true[:, i])

            residuals = np.abs(
                y_true[valid_mask, i]
                - y_pred[valid_mask, i]
            )

            n_cal = len(residuals)

            if n_cal == 0:
                raise ValueError(
                    f"No non-missing validation targets are available for "
                    f"{target_name}."
                )

            # Finite-sample split conformal quantile
            quantile_level = np.ceil(
                (n_cal + 1) * (1.0 - self.alpha)
            ) / n_cal

            quantile_level = min(
                1.0,
                max(0.0, quantile_level)
            )

            # "higher" gives the conservative empirical quantile needed
            # for split conformal calibration.
            self.q_hat[target_name] = float(
                np.quantile(
                    residuals,
                    quantile_level,
                    method="higher"
                )
            )

    def predict_intervals(self, y_pred: np.ndarray):

        intervals = {}

        for i, target_name in enumerate(STRENGTH_TARGETS):

            margin = self.q_hat[target_name]

            intervals[target_name] = {
                "point": y_pred[:, i],
                "lower": np.clip(
                    y_pred[:, i] - margin,
                    0.0,
                    None
                ),
                "upper": y_pred[:, i] + margin,
                "margin": margin
            }

        return intervals

In [18]:
# ==============================================================================
# 8. STAGE 1: VIRTUAL SENSING
# ==============================================================================

def train_virtual_sensors(
    X_train,
    y_train,
    X_valid,
    y_valid,
    X_test
):
    """
    Train LightGBM virtual sensing models for Er and UPV.

    To reduce leakage into Stage 2:
        - training virtual features are generated using K-fold OOF predictions
        - validation/test predictions come from models trained on the full train set
    """

    print("\n" + "=" * 80)
    print("STAGE 1: VIRTUAL SENSING")
    print("=" * 80)

    n_ndt = len(NDT_TARGET_INDICES)

    virtual_train_ndt = np.zeros(
        (len(X_train), n_ndt),
        dtype=np.float64
    )

    virtual_valid_ndt = np.zeros(
        (len(X_valid), n_ndt),
        dtype=np.float64
    )

    virtual_test_ndt = np.zeros(
        (len(X_test), n_ndt),
        dtype=np.float64
    )

    stage1_models = {}

    kfold = KFold(
        n_splits=5,
        shuffle=True,
        random_state=SEED
    )

    for output_idx, target_idx in enumerate(NDT_TARGET_INDICES):

        target_name = NDT_TARGETS[output_idx]

        y_ndt_train = y_train[:, target_idx]
        y_ndt_valid = y_valid[:, target_idx]

        # ------------------------------------------------------------------
        # OOF predictions for Stage-2 training features
        # ------------------------------------------------------------------

        for fold, (fit_idx, holdout_idx) in enumerate(kfold.split(X_train), start=1):

            observed_fit = ~np.isnan(y_ndt_train[fit_idx])

            if observed_fit.sum() == 0:
                raise ValueError(
                    f"No valid training labels for {target_name} in fold {fold}."
                )

            fold_model = lgb.LGBMRegressor(
                n_estimators=150,
                learning_rate=0.05,
                random_state=SEED + fold,
                verbose=-1
            )

            fold_model.fit(
                X_train[fit_idx][observed_fit],
                y_ndt_train[fit_idx][observed_fit]
            )

            virtual_train_ndt[
                holdout_idx,
                output_idx
            ] = fold_model.predict(
                X_train[holdout_idx]
            )

        # ------------------------------------------------------------------
        # Final Stage-1 model trained on all observed training labels
        # ------------------------------------------------------------------

        observed_train = ~np.isnan(y_ndt_train)

        final_model = lgb.LGBMRegressor(
            n_estimators=150,
            learning_rate=0.05,
            random_state=SEED,
            verbose=-1
        )

        final_model.fit(
            X_train[observed_train],
            y_ndt_train[observed_train]
        )

        stage1_models[target_name] = final_model

        virtual_valid_ndt[:, output_idx] = final_model.predict(
            X_valid
        )

        virtual_test_ndt[:, output_idx] = final_model.predict(
            X_test
        )

        # ------------------------------------------------------------------
        # Validation metric for Stage 1
        # ------------------------------------------------------------------

        observed_valid = ~np.isnan(y_ndt_valid)

        if observed_valid.sum() >= 2:

            valid_r2 = r2_score(
                y_ndt_valid[observed_valid],
                virtual_valid_ndt[observed_valid, output_idx]
            )

            valid_rmse = np.sqrt(
                mean_squared_error(
                    y_ndt_valid[observed_valid],
                    virtual_valid_ndt[observed_valid, output_idx]
                )
            )

            print(
                f"{target_name:<12} | "
                f"Validation R²: {valid_r2:.4f} | "
                f"RMSE: {valid_rmse:.4f}"
            )

        else:
            print(
                f"{target_name:<12} | "
                "Insufficient non-missing validation targets for R²."
            )

    return (
        stage1_models,
        virtual_train_ndt,
        virtual_valid_ndt,
        virtual_test_ndt
    )

In [19]:
# ==============================================================================
# 9. STAGE 2 MODEL TRAINING
# ==============================================================================

def train_stage2_model(
    X_train_stage2,
    y_train_strength,
    age_col_idx
):

    print("\n" + "=" * 80)
    print("STAGE 2: PHYSICS-INFORMED MULTI-TASK ANN")
    print("=" * 80)

    train_dataset = ConcreteMTLDataset(
        X_train_stage2,
        y_train_strength
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True
    )

    model = PhysicsInformedMTNN(
        input_dim=X_train_stage2.shape[1],
        hidden_dim=128,
        dropout_rate=0.2
    ).to(device)

    optimizer = optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY
    )

    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=EPOCHS
    )

    for epoch in range(1, EPOCHS + 1):

        model.train()

        total_loss = 0.0

        for batch_x, batch_y, batch_mask in train_loader:

            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)
            batch_mask = batch_mask.to(device)

            # Required for derivative-based monotonicity penalty.
            batch_x.requires_grad_(True)

            optimizer.zero_grad()

            predictions = model(batch_x)

            loss = compute_physics_loss(
                predictions,
                batch_y,
                batch_mask,
                batch_x,
                age_col_idx
            )

            loss.backward()
            optimizer.step()

            total_loss += loss.item() * len(batch_x)

        scheduler.step()

        if epoch == 1 or epoch % 30 == 0 or epoch == EPOCHS:

            average_loss = total_loss / len(train_dataset)

            print(
                f"Epoch {epoch:03d}/{EPOCHS} | "
                f"Physics Composite Loss: {average_loss:.6f}"
            )

    return model


# ==============================================================================
# 10. INFERENCE
# ==============================================================================

def predict(model, X):

    model.eval()

    X_tensor = torch.tensor(
        X,
        dtype=torch.float32
    ).to(device)

    with torch.no_grad():
        predictions = model(X_tensor).cpu().numpy()

    return predictions


# ==============================================================================
# 11. FINAL EVALUATION
# ==============================================================================

def evaluate_test_set(
    y_test,
    test_preds,
    test_intervals,
    conformal
):

    print("\n" + "=" * 80)
    print("FINAL TEST-SET EVALUATION")
    print("=" * 80)

    for i, target_name in enumerate(STRENGTH_TARGETS):

        actuals = y_test[:, i]

        valid_idx = ~np.isnan(actuals)

        y_true = actuals[valid_idx]
        y_pred = test_preds[valid_idx, i]

        if len(y_true) == 0:
            print(
                f"{target_name}: No observed test targets available."
            )
            continue

        lower = test_intervals[target_name]["lower"][valid_idx]
        upper = test_intervals[target_name]["upper"][valid_idx]

        rmse = np.sqrt(
            mean_squared_error(
                y_true,
                y_pred
            )
        )

        mae = mean_absolute_error(
            y_true,
            y_pred
        )

        if len(y_true) >= 2:
            r2 = r2_score(
                y_true,
                y_pred
            )
        else:
            r2 = np.nan

        covered = (
            (y_true >= lower)
            & (y_true <= upper)
        )

        picp = np.mean(covered) * 100.0
        mpiw = np.mean(upper - lower)

        print(
            f"Target: {target_name.upper():<8} | "
            f"R²: {r2:.4f} | "
            f"RMSE: {rmse:.4f} MPa | "
            f"MAE: {mae:.4f} MPa"
        )

        print(
            f"         95% Conformal Coverage: {picp:.2f}% | "
            f"Mean Interval Width: {mpiw:.4f} MPa | "
            f"Margin: +/-{conformal.q_hat[target_name]:.4f} MPa"
        )

        print("-" * 80)

In [20]:
# ==============================================================================
# 12. COMPLETE END-TO-END PIPELINE
# ==============================================================================

def run_intelligent_concrete_pipeline():

    print("=" * 80)
    print(
        "TWO-STAGE VIRTUAL SENSING + "
        "PHYSICS-INFORMED MULTI-TASK ANN + "
        "CONFORMAL CALIBRATION"
    )
    print("=" * 80)

    # ----------------------------------------------------------------------
    # 12.1 Load already-created train / validation / test partitions
    # ----------------------------------------------------------------------

    (
        X_train,
        y_train,
        X_valid,
        y_valid,
        X_test,
        y_test
    ) = load_npz_data()

    validate_dataset_shapes(
        X_train,
        y_train,
        X_valid,
        y_valid,
        X_test,
        y_test
    )

    # ----------------------------------------------------------------------
    # 12.2 Scale input features using TRAINING DATA ONLY
    #
    # If your NPZ inputs are already standardized, you may replace these
    # transformed arrays with X_train, X_valid and X_test directly.
    # ----------------------------------------------------------------------

    input_scaler = StandardScaler()

    X_train_scaled = input_scaler.fit_transform(
        X_train
    )

    X_valid_scaled = input_scaler.transform(
        X_valid
    )

    X_test_scaled = input_scaler.transform(
        X_test
    )

    # Convert negative age index to its absolute index.
    if AGE_INPUT_INDEX < 0:
        age_col_idx = X_train_scaled.shape[1] + AGE_INPUT_INDEX
    else:
        age_col_idx = AGE_INPUT_INDEX

    if not 0 <= age_col_idx < X_train_scaled.shape[1]:
        raise ValueError(
            f"AGE_INPUT_INDEX={AGE_INPUT_INDEX} is invalid for "
            f"{X_train_scaled.shape[1]} input features."
        )

    # ----------------------------------------------------------------------
    # 12.3 Stage 1: virtual sensing
    # ----------------------------------------------------------------------

    (
        stage1_models,
        virtual_train_ndt,
        virtual_valid_ndt,
        virtual_test_ndt
    ) = train_virtual_sensors(
        X_train_scaled,
        y_train,
        X_valid_scaled,
        y_valid,
        X_test_scaled
    )

    # ----------------------------------------------------------------------
    # 12.4 Scale virtual NDT variables using TRAINING DATA ONLY
    # ----------------------------------------------------------------------

    ndt_scaler = StandardScaler()

    scaled_virtual_train_ndt = ndt_scaler.fit_transform(
        virtual_train_ndt
    )

    scaled_virtual_valid_ndt = ndt_scaler.transform(
        virtual_valid_ndt
    )

    scaled_virtual_test_ndt = ndt_scaler.transform(
        virtual_test_ndt
    )

    # ----------------------------------------------------------------------
    # 12.5 Construct Stage-2 input matrices
    # ----------------------------------------------------------------------

    X_train_stage2 = np.hstack(
        [
            X_train_scaled,
            scaled_virtual_train_ndt
        ]
    )

    X_valid_stage2 = np.hstack(
        [
            X_valid_scaled,
            scaled_virtual_valid_ndt
        ]
    )

    X_test_stage2 = np.hstack(
        [
            X_test_scaled,
            scaled_virtual_test_ndt
        ]
    )

    print("\n[INFO] Stage-2 input dimensions")
    print(f"Train      : {X_train_stage2.shape}")
    print(f"Validation : {X_valid_stage2.shape}")
    print(f"Test       : {X_test_stage2.shape}")

    # ----------------------------------------------------------------------
    # 12.6 Extract strength targets
    # ----------------------------------------------------------------------

    y_train_strength = y_train[
        :,
        STRENGTH_TARGET_INDICES
    ]

    y_valid_strength = y_valid[
        :,
        STRENGTH_TARGET_INDICES
    ]

    y_test_strength = y_test[
        :,
        STRENGTH_TARGET_INDICES
    ]

    # ----------------------------------------------------------------------
    # 12.7 Train Stage-2 model
    #
    # The age column remains in its original location because the two virtual
    # NDT features were appended at the END of the Stage-2 input matrix.
    # ----------------------------------------------------------------------

    model = train_stage2_model(
        X_train_stage2,
        y_train_strength,
        age_col_idx
    )

    # ----------------------------------------------------------------------
    # 12.8 Validation and test inference
    # ----------------------------------------------------------------------

    valid_preds = predict(
        model,
        X_valid_stage2
    )

    test_preds = predict(
        model,
        X_test_stage2
    )

    # ----------------------------------------------------------------------
    # 12.9 Conformal calibration
    #
    # The validation partition acts as the calibration partition.
    # The test set remains untouched until final evaluation.
    # ----------------------------------------------------------------------

    print("\n" + "=" * 80)
    print("CONFORMAL CALIBRATION USING VALIDATION DATA")
    print("=" * 80)

    conformal = MultiTaskConformalCalibrator(
        alpha=0.05
    )

    conformal.calibrate(
        y_valid_strength,
        valid_preds
    )

    for target_name in STRENGTH_TARGETS:
        print(
            f"{target_name:<8} | "
            f"95% conformal margin = "
            f"+/-{conformal.q_hat[target_name]:.4f}"
        )

    test_intervals = conformal.predict_intervals(
        test_preds
    )

    # ----------------------------------------------------------------------
    # 12.10 Final test evaluation
    # ----------------------------------------------------------------------

    evaluate_test_set(
        y_test_strength,
        test_preds,
        test_intervals,
        conformal
    )

    return {
        "model": model,
        "stage1_models": stage1_models,
        "input_scaler": input_scaler,
        "ndt_scaler": ndt_scaler,
        "conformal": conformal,
        "validation_predictions": valid_preds,
        "test_predictions": test_preds,
        "test_intervals": test_intervals
    }


# ==============================================================================
# 13. RUN
# ==============================================================================

if __name__ == "__main__":
    results = run_intelligent_concrete_pipeline()

TWO-STAGE VIRTUAL SENSING + PHYSICS-INFORMED MULTI-TASK ANN + CONFORMAL CALIBRATION
DATASET LOADED
Train inputs       : (1326, 4)
Train targets      : (1326,)
Validation inputs  : (310, 4)
Validation targets : (310,)
Test inputs        : (2784, 4)
Test targets       : (2784,)


ValueError: All target arrays must be 2-dimensional.